# Hugging Face Data sets

**Hugging Face** has evolved far beyond a repository for transformer models; it is now the central hub for the open-source AI and machine learning ecosystem. For a data scientist, pushing a Parquet file to a Hugging Face Dataset repository turns a static file into a highly accessible, collaborative, and optimized ML asset.

Here is why it's a smart move for your workflow:

* **Seamless Pandas Integration:** Hugging Face integrates flawlessly with your existing workflow. Once your Parquet file is in an HF repo, you or your teammates can load it directly into a local Python environment using a single line of code (`load_dataset`), and convert it straight into a familiar **pandas DataFrame** with `.to_pandas()`.
* **Memory-Efficient Streaming:** Because Parquet is a columnar format and Hugging Face supports streaming, you can load and inspect massive datasets without running out of local RAM or waiting for giant files to fully download.
* **Version Control & Collaboration:** It essentially acts as "GitHub for data." You get full commit histories, automatic web previews of your data rows, and built-in discussion tabs, making it incredibly easy to share clean features or training sets across your team or the wider data science community.


In [ ]:
import os
import duckdb
from google.colab import userdata


## Hugging Face



### Create account

- https://huggingface.co/join


### Create a data set repo

- https://huggingface.co/new-dataset

Name it `example`.


### Create a Personal Access Token ( PAT )

- https://huggingface.co/settings/tokens/new?tokenType=fineGrained

- Add repo to token



### Save PAT and repo name in Colab Secrets Vault

- Create a secret named `hf_token`

- Create a secret named `hf_account`


## Configure settings and authenticate to Hugging Face


In [ ]:
hf_account = os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account


In [ ]:
hf_repo = os.environ["HF_REPO"] = "example"
hf_repo


In [ ]:
hf_file = os.environ["HF_FILE"] = "example.parquet"
hf_file


In [ ]:
# Retrieve the token from Colab's secret manager
_ = hf_token = os.environ["HF_TOKEN"] = userdata.get('hf_token')
f"{_[:3]} ... {_[-4:]}"


In [ ]:
# Log in automatically without an interactive prompt
! hf auth login --token ${HF_TOKEN}


## Pull a parquet file from Hugging Face


In [ ]:
hf_url = "https://huggingface.co/datasets/rwcitek/irs-teos/resolve/main/data-download-990n.parquet"
hf_url


In [ ]:
! curl -L -o {hf_file} {hf_url}


In [ ]:
! ls -la --si {hf_file}


In [ ]:
# Query the local parquet
query = (f'''
  select count(1) as count
  from "{hf_file}"
  ;
''')
print(query)
duckdb.sql(query).df()


## Verify that the parquet file can be queried using DuckDB


In [ ]:
# Query the remote parquet
query = (f'''
  select count(1) as count
  from "{hf_url}"
  ;
''')
print(query)
duckdb.sql(query).df()


In [ ]:
# Fetch first 10 rows and display as a data frame
query = (f'''
  select *
  from "{hf_file}"
  limit 10
  ;
''')
print(query)
duckdb.sql(query).df()


## Push the parquet file to Hugging Face


In [ ]:
%%capture hf_upload
%%bash

TARGET_REPO="${HF_ACCOUNT}/${HF_REPO}"
LOCAL_SOURCE="./${HF_FILE}"
REMOTE_DESTINATION="./${HF_FILE}"

# Comment out the echo to make this live
echo DRY-RUN: \
hf upload \
  --type dataset \
  "${TARGET_REPO}" \
  "${LOCAL_SOURCE}" \
  "${REMOTE_DESTINATION}"


In [ ]:
print(hf_upload.stdout)


## Query the parquet on Hugging Face using DuckDB


In [ ]:
# Construct the Hugging Face URL for your dataset file
hf_url = f"https://huggingface.co/datasets/{hf_account}/{hf_repo}/resolve/main/{hf_file}"
hf_url


In [ ]:
# Verify that the URL works
! curl -s -L -I "{hf_url}" | tr -d '\r'


In [ ]:
# Query the remote parquet
query = (f'''
  select count(1) as count
  from "{hf_url}"
  ;
''')
print(query)
duckdb.sql(query).df()


In [ ]:
# Query the schema of the remote parquet
query = (f'''
  describe
  select *
  from "{hf_url}"
  ;
''')
print(query)
duckdb.sql(query).df()
